# AdventureWorks — Exercise 1: Core ETL Pipeline

**Course:** Data Engineering  
**Notebook:** 01 — Building the Core Data Pipeline  
**Assistant:** Antigravity  

---

## What This Notebook Covers

| # | Section | Purpose |
|---|---------|----------|
| 1 | Configuration & Setup | Verify paths and import libraries |
| 2 | Source File Check | Confirm all CSV files exist |
| 3 | Data Extraction | Load all 4 tables with Pandas |
| 4 | Schema Display | Inspect column names, types, shapes |
| 5 | Record Counts | How many rows per table? |
| 6 | Sample Records | Preview actual data |
| 7 | Raw Staging | Persist raw data to staging/raw/ |

> **Architecture Note:** This notebook uses a **Full Refresh** strategy.  
> Every time you run it, the data is read fresh from the source CSVs.  
> No old data is carried forward.


---
## Section 1 — Configuration & Setup

**Objective:**  
Import required libraries and load the project configuration.  
The configuration file (`src/config.py`) is the **single source of truth** for all paths.

> **Why a config file?**  
> Instead of hard-coding `C:\Users\Kevin\Data\...` everywhere, we define `DATA_PATH` once.  
> If you move your CSV files, you only need to change one line in `config.py`.


In [ ]:
# ── Standard Library ──────────────────────────────────────────
import sys
import os
from pathlib import Path
from datetime import datetime

# ── Third-Party ───────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Notebook Display Settings ─────────────────────────────────
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 120)

# ── Add project /src to Python path ──────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent          # AdventureWorks_DataEngineering/
SRC_DIR      = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

# ── Import project config ─────────────────────────────────────
from config import (
    DATA_PATH, SOURCE_FILES,
    CSV_SEPARATOR,
    STAGING_RAW,
    PRODUCT_COLUMNS, CUSTOMER_COLUMNS,
    SALESORDERHEADER_COLUMNS, SALESORDERDETAIL_COLUMNS,
    ensure_directories,
)

print('✓ Libraries imported')
print(f'✓ Project Root : {PROJECT_ROOT}')
print(f'✓ Data Path    : {DATA_PATH}')
print(f'✓ Staging Raw  : {STAGING_RAW}')

**Explanation:**  
- We add the `/src` folder to Python's module search path so `import config` works from the notebook.  
- `PROJECT_ROOT` is computed dynamically — this notebook works on any machine without path changes.  
- `DATA_PATH` is imported from `config.py`. Change it there, not here.


---
## Section 2 — Source File Check

**Objective:**  
Before reading any data, verify that all four required CSV files exist at the configured path.  
This prevents cryptic errors later in the pipeline.


In [ ]:
print('=' * 65)
print('  SOURCE FILE VERIFICATION')
print('=' * 65)
print(f'  Configured DATA_PATH: {DATA_PATH}\n')

all_found = True
file_info = []

for name, path in SOURCE_FILES.items():
    exists   = path.exists()
    size_mb  = round(path.stat().st_size / 1_048_576, 2) if exists else 0
    status   = '✓ FOUND' if exists else '✗ MISSING'
    color    = '' if exists else '[!] '
    all_found = all_found and exists
    file_info.append({'Table': name, 'File': path.name, 'Status': status, 'Size (MB)': size_mb})
    print(f'  {status}  {name:20s}  {path.name:30s}  {size_mb:>6} MB')

print()
if all_found:
    print('  ✅ All source files found. Ready to extract.')
else:
    print('  ❌ One or more files are MISSING.')
    print('  → Open src/config.py and update DATA_PATH to the correct folder.')
print('=' * 65)

**Explanation:**  
- We iterate over `SOURCE_FILES` (defined in `config.py`) which maps table names to file paths.  
- `path.exists()` checks whether the file is present on disk.  
- `path.stat().st_size` gives the file size in bytes; we convert to MB for readability.  
- If any file is missing, the pipeline should stop here rather than produce partial results.


---
## Section 3 — Data Extraction

**Objective:**  
Read all four AdventureWorks CSV files into Pandas DataFrames.

**Key facts about the source files:**  
- Separator: **TAB** (`\t`) — not comma-separated  
- No header row — the first row is already data  
- We assign column names manually using the lists in `config.py`


In [ ]:
def read_source_csv(name: str, columns: list) -> pd.DataFrame:
    """
    Read a single AdventureWorks CSV (tab-separated, no header).
    
    Parameters
    ----------
    name    : key in SOURCE_FILES dict, e.g. 'product'
    columns : list of column names to assign positionally
    
    Returns
    -------
    pd.DataFrame
    """
    path = SOURCE_FILES[name]
    
    df = pd.read_csv(
        path,
        sep=CSV_SEPARATOR,    # Tab separator
        header=None,          # No header row in file
        names=columns,        # Assign column names
        low_memory=False,
        encoding='utf-8',
        on_bad_lines='warn',  # Log bad lines, don't crash
    )
    
    # Add metadata columns for lineage
    df['_source_file'] = path.name
    df['_ingested_at'] = datetime.utcnow().isoformat(timespec='seconds')
    
    return df

print('✓ read_source_csv() helper defined')

In [ ]:
print('Extracting all source tables…\n')

start_time = datetime.utcnow()

# ── Extract each table ────────────────────────────────────────
df_product            = read_source_csv('product',          PRODUCT_COLUMNS)
df_customer           = read_source_csv('customer',         CUSTOMER_COLUMNS)
df_salesorderheader   = read_source_csv('salesorderheader', SALESORDERHEADER_COLUMNS)
df_salesorderdetail   = read_source_csv('salesorderdetail', SALESORDERDETAIL_COLUMNS)

end_time = datetime.utcnow()
elapsed  = (end_time - start_time).total_seconds()

print(f'  ✓ Product            → {len(df_product):>6,} records  |  {len(df_product.columns)} columns')
print(f'  ✓ Customer           → {len(df_customer):>6,} records  |  {len(df_customer.columns)} columns')
print(f'  ✓ SalesOrderHeader   → {len(df_salesorderheader):>6,} records  |  {len(df_salesorderheader.columns)} columns')
print(f'  ✓ SalesOrderDetail   → {len(df_salesorderdetail):>6,} records  |  {len(df_salesorderdetail.columns)} columns')
print(f'\n  Extraction complete in {elapsed:.2f} seconds.')

**Explanation:**  
- `pd.read_csv()` with `sep='\t'` handles tab-separated files.  
- `header=None` tells Pandas the file has no column header row.  
- `names=columns` assigns column names in positional order (column 0 = first name, etc.).  
- `_source_file` and `_ingested_at` are **lineage metadata** — they record where each row came from and when it was loaded. This is a best practice in data engineering.


---
## Section 4 — Schema Display

**Objective:**  
Inspect the structure (schema) of each table: column names, data types, and null counts.

Understanding the schema is the **first step** in any data engineering project.  
You need to know what you have before you can transform it.


In [ ]:
def show_schema(df: pd.DataFrame, table_name: str) -> None:
    """
    Display a formatted schema for a DataFrame.
    Shows: column index, name, data type, non-null count, null count, null%.
    """
    data_cols = [c for c in df.columns if not c.startswith('_')]   # exclude metadata
    
    print(f'\n{"=" * 70}')
    print(f'  SCHEMA: {table_name}')
    print(f'  Rows: {len(df):,}   |   Columns: {len(data_cols)}')
    print(f'  {"-" * 66}')
    print(f'  {"#":<4} {"Column Name":<35} {"Dtype":<15} {"Non-Null":>8} {"Nulls":>6} {"Null%":>6}')
    print(f'  {"-" * 66}')
    
    for i, col in enumerate(data_cols):
        non_null  = df[col].notna().sum()
        nulls     = df[col].isna().sum()
        null_pct  = round(nulls / len(df) * 100, 1) if len(df) > 0 else 0.0
        dtype_str = str(df[col].dtype)
        print(f'  {i:<4} {col:<35} {dtype_str:<15} {non_null:>8,} {nulls:>6,} {null_pct:>5.1f}%')
    
    print(f'  {"=" * 66}')

print('✓ show_schema() helper defined')

In [ ]:
# ── Product Schema ────────────────────────────────────────────
show_schema(df_product, 'Product')

In [ ]:
# ── Customer Schema ───────────────────────────────────────────
show_schema(df_customer, 'Customer')

In [ ]:
# ── SalesOrderHeader Schema ───────────────────────────────────
show_schema(df_salesorderheader, 'SalesOrderHeader')

In [ ]:
# ── SalesOrderDetail Schema ───────────────────────────────────
show_schema(df_salesorderdetail, 'SalesOrderDetail')

**Explanation:**  
- `df.dtypes` returns a Series with column names as the index and data types as values.  
- `df[col].isna().sum()` counts missing (NaN/None) values per column.  
- At this stage, all numeric-looking columns are still `object` (string) because we haven't performed type conversion yet. That happens in the Transformation step (Exercise 1, Step 2).  
- High null percentages in columns like `Color`, `Size`, `Weight` are expected — many AdventureWorks products don't have these attributes.


---
## Section 5 — Record Counts

**Objective:**  
Summarize the size of each table at a glance.

Record counts are the simplest but most important data quality check —  
if a table suddenly has 0 rows or 10× the expected rows, something is wrong.


In [ ]:
import datetime as dt

tables = {
    'Product':          df_product,
    'Customer':         df_customer,
    'SalesOrderHeader': df_salesorderheader,
    'SalesOrderDetail': df_salesorderdetail,
}

print('\n' + '=' * 55)
print('  RECORD COUNT SUMMARY')
print('=' * 55)
print(f'  {"Table":<22} {"Rows":>10}  {"Columns":>8}  {"Memory":>10}')
print('  ' + '-' * 51)

total_rows = 0
for name, df in tables.items():
    rows    = len(df)
    cols    = len(df.columns)
    mem_kb  = round(df.memory_usage(deep=True).sum() / 1024, 1)
    total_rows += rows
    print(f'  {name:<22} {rows:>10,}  {cols:>8}  {mem_kb:>8.1f} KB')

print('  ' + '-' * 51)
print(f'  {"TOTAL ROWS":<22} {total_rows:>10,}')
print('=' * 55)
print(f'  Extracted at: {datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")} UTC')

**Explanation:**  
- `len(df)` returns the number of rows.  
- `df.memory_usage(deep=True).sum()` returns the total memory used by the DataFrame in bytes.  
- `deep=True` is important for object columns — without it, string columns are under-counted.  
- In a production pipeline, record counts are logged at each stage. A sudden drop (e.g., source file truncated) should trigger an alert.


---
## Section 6 — Sample Records

**Objective:**  
Preview the first few rows of each table to visually confirm that  
column names were assigned correctly and data looks reasonable.

> **Tip:** Always look at sample data before writing any transformation.  
> Real-world data is messy — looking at samples surfaces surprises early.


In [ ]:
# Helper to display samples without the internal metadata columns
def show_sample(df: pd.DataFrame, table_name: str, n: int = 5) -> None:
    data_cols = [c for c in df.columns if not c.startswith('_')]
    print(f'\n── {table_name} — first {n} rows ──')
    display(df[data_cols].head(n))

show_sample(df_product, 'Product')

In [ ]:
show_sample(df_customer, 'Customer')

In [ ]:
show_sample(df_salesorderheader, 'SalesOrderHeader')

In [ ]:
show_sample(df_salesorderdetail, 'SalesOrderDetail')

In [ ]:
# Statistical summary of key numeric-looking columns in Product
print('\n── Product — Basic Statistics for key columns ──')
key_cols = ['ProductID', 'SafetyStockLevel', 'ReorderPoint',
            'StandardCost', 'ListPrice', 'DaysToManufacture']
# Convert to numeric first (they come in as strings)
product_stats = df_product[key_cols].apply(pd.to_numeric, errors='coerce')
display(product_stats.describe().round(2))

**Explanation:**  
- `display()` renders DataFrames as nicely formatted HTML tables inside Jupyter.  
- We filter out `_source_file` and `_ingested_at` columns using `not c.startswith('_')`.  
- `pd.to_numeric(errors='coerce')` converts values to numbers, turning non-numeric values to `NaN` rather than raising an error — useful for spotting data quality issues.  
- `describe()` gives count, mean, std, min, max, and quartiles — a fast way to spot outliers and range issues.


---
## Section 7 — Raw Staging

**Objective:**  
Persist the raw extracted DataFrames to `staging/raw/` in **Parquet format**.

**Why Parquet?**  
- Faster to read than CSV (columnar storage)  
- Preserves data types  
- Much smaller file size than CSV  
- Industry standard in data engineering

**Why a raw staging layer?**  
- It preserves the exact source data before any transformation.  
- If a transformation later proves wrong, you can re-run from staging without re-reading CSVs.  
- Acts as an audit trail of what was ingested and when.

**Full Refresh behavior:**  
> Every time the pipeline runs, staging/raw/ files are **overwritten** completely.  
> They always represent the current state of the source CSVs.


In [ ]:
import pyarrow   # Required by pandas for Parquet write

# Ensure staging directories exist
ensure_directories()

# Map: (DataFrame, raw table name)
raw_tables = [
    (df_product,          'raw_product'),
    (df_customer,         'raw_customer'),
    (df_salesorderheader, 'raw_salesorderheader'),
    (df_salesorderdetail, 'raw_salesorderdetail'),
]

print('Writing raw staging files…\n')
print(f'  Target directory: {STAGING_RAW}\n')

staged_paths = {}
for df, table_name in raw_tables:
    out_path = STAGING_RAW / f'{table_name}.parquet'
    
    # FULL REFRESH: always overwrite, never append
    df.to_parquet(out_path, index=False, engine='pyarrow')
    
    size_kb  = round(out_path.stat().st_size / 1024, 1)
    staged_paths[table_name] = out_path
    print(f'  ✓ {table_name:<30}  →  {out_path.name:<40} ({size_kb:>8.1f} KB)')

print('\n  Raw staging complete.')

In [ ]:
# Verify: Read back one file and confirm row count matches
print('\n── Verification: Read-back check ──')
print('(Confirms staging files were written correctly)\n')

verify_pairs = [
    ('raw_product',          df_product,          'Product'),
    ('raw_customer',         df_customer,          'Customer'),
    ('raw_salesorderheader', df_salesorderheader,  'SalesOrderHeader'),
    ('raw_salesorderdetail', df_salesorderdetail,  'SalesOrderDetail'),
]

all_ok = True
for table_name, original_df, label in verify_pairs:
    path     = STAGING_RAW / f'{table_name}.parquet'
    reloaded = pd.read_parquet(path)
    match    = len(reloaded) == len(original_df)
    status   = '✓ MATCH' if match else '✗ MISMATCH'
    all_ok   = all_ok and match
    print(f'  {status}  {label:<22}  original={len(original_df):>6,}  staged={len(reloaded):>6,}')

print()
if all_ok:
    print('  ✅ All raw staging files verified successfully.')
else:
    print('  ❌ Verification FAILED — check file write permissions and disk space.')

**Explanation:**  
- `df.to_parquet(path, index=False)` saves the DataFrame without writing the Pandas row index.  
- `engine='pyarrow'` uses the Apache Arrow library for fast, efficient Parquet I/O.  
- The verification step reads each Parquet file back and compares row counts with the original DataFrame. This is a basic **data quality gate** — a pattern you'll use throughout the pipeline.  
- Notice how Parquet files are significantly smaller than the source CSV files while storing the same data.


---
## Section 8 — Step 1 Summary

**Objective:**  
Print a final summary of everything accomplished in this step.


In [ ]:
print('\n' + '=' * 60)
print('  STEP 1 COMPLETE — CORE ETL SUMMARY')
print('=' * 60)
print()
print('  Source Path   :', DATA_PATH)
print('  Staging (Raw) :', STAGING_RAW)
print()
print('  ┌─────────────────────┬────────────┬──────────┐')
print('  │ Table               │ Records    │ Staging  │')
print('  ├─────────────────────┼────────────┼──────────┤')
print(f'  │ Product             │ {len(df_product):>10,} │ Parquet  │')
print(f'  │ Customer            │ {len(df_customer):>10,} │ Parquet  │')
print(f'  │ SalesOrderHeader    │ {len(df_salesorderheader):>10,} │ Parquet  │')
print(f'  │ SalesOrderDetail    │ {len(df_salesorderdetail):>10,} │ Parquet  │')
print('  └─────────────────────┴────────────┴──────────┘')
print()
print('  What was accomplished:')
print('    ✓ Configuration loaded from src/config.py')
print('    ✓ All 4 source CSV files verified')
print('    ✓ Data extracted with correct column names')
print('    ✓ Schemas displayed')
print('    ✓ Record counts verified')
print('    ✓ Sample records previewed')
print('    ✓ Raw staging files written to staging/raw/')
print('    ✓ Staging files verified (read-back check)')
print()
print('  Next: Step 2 — Data Transformation & Validation')
print('=' * 60)

---
## Concepts Learned in This Notebook

| Concept | What It Means |
|---------|---------------|
| **Data Extraction** | Reading raw data from source systems (files, databases, APIs) |
| **Configuration-Driven Design** | Using a config file to avoid hard-coded paths |
| **Tab-Separated Values (TSV)** | CSV variant using tab as delimiter instead of comma |
| **Schema** | The structure of a table: column names and data types |
| **Null / Missing Values** | Fields with no value — important to detect early |
| **Raw Staging Layer** | A copy of source data before transformation — preserves original state |
| **Parquet Format** | Columnar binary format — faster and smaller than CSV |
| **Full Refresh** | Every run starts from scratch — no dependency on previous runs |
| **Lineage Metadata** | `_source_file`, `_ingested_at` — track where data came from |
| **Verification / Data Quality Gate** | Checking outputs match expectations before proceeding |

---
*Next notebook:* `02_schema_design.ipynb` — OLTP and Star Schema design  
*Developed with:* **Antigravity** — AI Coding Assistant
